In [1]:
import os
import json
import pandas as pd

from pathlib import Path


# Get GEO_ID from folder name (same pattern as prepare_data.ipynb)
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))


description_path = Path('description.json')
preprocessed_csv_path = Path(f"{GEO_ID}_preprocessed.csv")

if not description_path.exists():
    raise FileNotFoundError(
        "description.json not found. Please run 'prepare_data.ipynb' first to generate it."
    )

if not preprocessed_csv_path.exists():
    raise FileNotFoundError(
        f"{preprocessed_csv_path} not found. Please run 'preprocess_data.ipynb' first to generate it."
    )

with open(description_path) as f:
    summary_data = json.load(f)

df = pd.read_csv(preprocessed_csv_path)

print("Data contains", df.shape[0], "samples and", df.shape[1], "columns.")
print("First few rows of the data:")
df.head()


Data contains 34 samples and 9790 columns.
First few rows of the data:


,OS (days),Alive at analysis?,ID,Sex,Age,CA19-9 (U/mL),CEA (ng/mL),sample_title,sample type,cell type,...,COX2,ATP8,ATP6,COX3,ND3,ND4L,ND4,ND5,ND6,CYTB
0,601.0,1.0,locPDAC01,M,82.0,224.0,3.6,locPDAC01,primary human sample,purified circulating tumor cells,...,8.786438,7.488502,8.021165,7.581757,6.736391,7.178738,8.729786,8.858936,6.981255,8.318217
1,317.0,1.0,locPDAC02,F,49.0,566.0,1.1,locPDAC02,primary human sample,purified circulating tumor cells,...,6.916434,4.912205,6.026992,6.394777,4.322666,4.879872,6.906090,6.600385,5.738722,6.414802
2,208.0,1.0,locPDAC03,M,70.0,40.0,4.7,locPDAC03,primary human sample,purified circulating tumor cells,...,8.777108,6.799338,8.114119,8.431929,6.507639,6.710036,8.597269,7.912436,7.035319,8.299621
3,1139.0,1.0,locPDAC04,M,72.0,82.0,5.0,locPDAC04,primary human sample,purified circulating tumor cells,...,6.366303,4.437187,5.532977,6.174563,3.070975,4.117468,6.169198,5.700986,4.853849,6.122393
4,1016.0,1.0,locPDAC05,M,65.0,1943.0,4.8,locPDAC05,primary human sample,purified circulating tumor cells,...,5.991699,4.108643,5.419751,5.947512,3.216698,3.862012,5.950981,5.232411,4.486394,5.650229


In [2]:
n_stats_written = 0

for idx, endpoint in enumerate(summary_data['survival-endpoints']):
    time_var = endpoint['time_var']['var_name']
    event_var = endpoint['event_var']['var_name']

    if time_var not in df.columns or event_var not in df.columns:
        abbrv = endpoint.get('abbrv', f'endpoint[{idx}]')
        print(
            f"Skipping stats for {abbrv}: columns not in preprocessed table "
            f"({time_var!r}, {event_var!r})."
        )
        endpoint.pop('stats', None)
        continue

    complete_mask = df[[time_var, event_var]].notnull().all(axis=1)
    n_complete = int(complete_mask.sum())
    n_incomplete = int(len(df) - n_complete)

    # Among samples with complete time + event data
    censored_count = int((complete_mask & (df[event_var] == 0.0)).sum())
    event_count = int((complete_mask & (df[event_var] == 1.0)).sum())

    censored_ratio = censored_count / n_complete

    print(f"Number of samples with incomplete data for {time_var} or {event_var}: {n_incomplete}")
    print(f"Number of samples with complete data for both variables: {n_complete}")
    print(f"Number of censored samples (event_var == 0.0): {censored_count}")
    print(f"Number of events (event_var == 1.0): {event_count}")
    pct = 100.0 * censored_ratio
    print(
        f"Censored ratio (among complete): {censored_count} / {n_complete} = "
        f"{censored_ratio:.3f} ({pct:.1f}%)"
    )

    # Refresh only this block on each run; leave all other endpoint fields unchanged
    endpoint['stats'] = {
        'n_incomplete': n_incomplete,
        'n_complete': n_complete,
        'n_censored': censored_count,
        'n_events': event_count,
        'censored_ratio': round(censored_ratio, 2),
    }
    n_stats_written += 1

with open(description_path, 'w', encoding='utf-8') as f:
    json.dump(summary_data, f, indent=4, ensure_ascii=False)
    f.write('\n')

print(
    f"Updated stats for {n_stats_written} of {len(summary_data['survival-endpoints'])} endpoint(s); "
    f"wrote {description_path.resolve()}"
)


Number of samples with incomplete data for OS (days) or Alive at analysis?: 1
Number of samples with complete data for both variables: 33
Number of censored samples (event_var == 0.0): 5
Number of events (event_var == 1.0): 28
Censored ratio (among complete): 5 / 33 = 0.152 (15.2%)
Updated stats for 1 of 1 endpoint(s); wrote /Users/jakakokosar/dev/phd/survival-rnaseq-data/datasets/geo/GSE144561/description.json
